# 05. 페르소나 연결 · 문단 분류 · 관측 상태 · 평가
PR #29의 세 번째 묶음입니다. 아래 순서를 그대로 따라갑니다.

1. 페르소나 결정 (직접 선택 또는 설문)
2. 산업 Agent: 문단 분류 기준선
3. 기술 Agent: 관측 상태 규칙
4. 평가: 페르소나 변경 전후 보존 확인
5. ML 확장 판단

DB·API 키 없이 실행됩니다. 위에서부터 차례로 실행하세요(메뉴 Run → Run All Cells).
예시 자료는 합성·모의 데이터입니다. 실제 기업 분석 결과가 아닙니다.

In [1]:
import sys
import tempfile
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'agent/neon_agents.py').exists()), None)
if ROOT is None:
    raise RuntimeError('압축 전체를 해제한 프로젝트/notebooks에서 실행하세요.')
sys.path.insert(0, str(ROOT))
pd.set_option('display.max_colwidth', 120)
print('프로젝트 위치 확인 완료')
from agent.basic_agents import AgentSettings, IndustryBusinessAgent, TechnicalAnalysisAgent, select_persona
from agent.neon_agents import execute_report
from agent.snapshots import mock_snapshot, reports_from_snapshot

snapshot = mock_snapshot()
report = reports_from_snapshot(snapshot)[0]
OPTIONS = {'reference_date': '2026-10-28', 'trading_dates': snapshot['mockTradingDates'], 'corporate_actions': []}


def run(agent_cls, domain, settings):
    gate, result = execute_report(report, agent_cls(settings=settings), domain, **OPTIONS)
    return gate, result

프로젝트 위치 확인 완료


## 1. 페르소나 결정
직접 선택은 사용자 확인으로 봅니다. 설문 결과는 사용자가 확인(confirmed=True)하기 전까지 L2·중립으로 표시합니다.

In [2]:
direct, notes = select_persona(literacy='L1', style='C')
print('직접 선택:', direct.persona_id, direct.status, notes)

knowledge = dict(zip([f'K{i}' for i in range(1, 9)], ['b', 'c', 'a', 'c', 'a', 'b', 'b', 'c']))
style = {'S1': 'A', 'S2': 'A', 'S3': 'A', 'S4': 'C'}
suggested, notes = select_persona(knowledge_answers=knowledge, style_answers=style)
confirmed, _ = select_persona(knowledge_answers=knowledge, style_answers=style, confirmed=True)
for name, profile in [('직접 선택', direct), ('설문 제안(미확인)', suggested), ('설문 확인', confirmed)]:
    s = AgentSettings.from_persona(profile, window=5)
    print(f'{name:12} {profile.persona_id} {profile.status:15} → 표현 {s.literacy}-{s.style}')

직접 선택: L1-C user_confirmed ['사용자 직접 선택']
직접 선택        L1-C user_confirmed  → 표현 L1-C
설문 제안(미확인)   L4-A suggested       → 표현 L2-U
설문 확인        L4-A user_confirmed  → 표현 L4-A


### 같은 결과, 다른 제시 방식
신중형(C)은 위험 절을, 적극형(A)은 기회 절을 먼저 보여줍니다. 어느 쪽도 다른 절을 빼지 않습니다.

In [3]:
from agent.persona_presentation import compare_personas

for persona in [('L1', 'C'), ('L4', 'A')]:
    settings = AgentSettings(window=5, literacy=persona[0], style=persona[1], persona_status='user_confirmed')
    _, result = run(IndustryBusinessAgent, 'industry_business_model', settings)
    plan = result['presentation']
    print(f"[{plan['personaId']}] {plan['literacy']['name']}·{plan['style']['name']} / 상세도 {plan['detail']}")
    print('  제시 순서:', ' → '.join(s['title'] for s in plan['sections']))
    print('  용어 풀이', len(plan['glossary']), '개 / 확인 질문:', plan['checkQuestions'][0])
display(pd.DataFrame(compare_personas(result)))

[L1-C] 입문·신중형 / 상세도 brief
  제시 순서: 사업 요약 → 위험 요인(하방 조건) → 자료 공백·한계 → 경쟁 5요인 후보 발췌 → 기회 요인(성립 조건) → 근거 원문 위치
  용어 풀이 6 개 / 확인 질문: 가장 큰 의존 위험(고객·원재료·규제)은 무엇이고 원문 근거가 있나요?
[L4-A] 고급·적극형 / 상세도 methodological
  제시 순서: 사업 요약 → 근거 원문 위치 → 자료 공백·한계 → 기회 요인(성립 조건) → 경쟁 5요인 후보 발췌 → 위험 요인(하방 조건)
  용어 풀이 0 개 / 확인 질문: 이 기회가 성립하려면 어떤 조건이 계속 유지되어야 하나요?


,페르소나,이해도,성향,상세도,제시 순서,용어 풀이,표시 주장 수,확인 질문
0,L1-A,입문,적극형,brief,사업 요약 → 기회 요인(성립 조건) → 경쟁 5요인 후보 발췌 → 위험 요인(하방 조건) → 근거 원문 위치 → 자료 공백·한계,6,3,이 기회가 성립하려면 어떤 조건이 계속 유지되어야 하나요? / 이 가설이 틀렸다고 판단할 근거(실패 조건)는 무엇인가요? / 모르는 용어가 있으면 용어 풀이를 먼저 확인하세요.
1,L1-C,입문,신중형,brief,사업 요약 → 위험 요인(하방 조건) → 자료 공백·한계 → 경쟁 5요인 후보 발췌 → 기회 요인(성립 조건) → 근거 원문 위치,6,3,가장 큰 의존 위험(고객·원재료·규제)은 무엇이고 원문 근거가 있나요? / 위험을 완화하는 근거는 원문에서 확인되나요? / 모르는 용어가 있으면 용어 풀이를 먼저 확인하세요.
2,L1-U,입문,중립(미확정),brief,사업 요약 → 기회 요인(성립 조건) → 위험 요인(하방 조건) → 경쟁 5요인 후보 발췌 → 근거 원문 위치 → 자료 공백·한계,6,3,기회와 위험 각각의 원문 근거가 충분한가요? / 모르는 용어가 있으면 용어 풀이를 먼저 확인하세요.
3,L2-A,기초,적극형,standard,사업 요약 → 기회 요인(성립 조건) → 경쟁 5요인 후보 발췌 → 위험 요인(하방 조건) → 근거 원문 위치 → 자료 공백·한계,6,3,이 기회가 성립하려면 어떤 조건이 계속 유지되어야 하나요? / 이 가설이 틀렸다고 판단할 근거(실패 조건)는 무엇인가요? / 각 수치가 어떤 원문·계산에서 나왔는지 근거 ID로 확인하세요.
4,L2-C,기초,신중형,standard,사업 요약 → 위험 요인(하방 조건) → 자료 공백·한계 → 경쟁 5요인 후보 발췌 → 기회 요인(성립 조건) → 근거 원문 위치,6,3,가장 큰 의존 위험(고객·원재료·규제)은 무엇이고 원문 근거가 있나요? / 위험을 완화하는 근거는 원문에서 확인되나요? / 각 수치가 어떤 원문·계산에서 나왔는지 근거 ID로 확인하세요.
5,L2-U,기초,중립(미확정),standard,사업 요약 → 기회 요인(성립 조건) → 위험 요인(하방 조건) → 경쟁 5요인 후보 발췌 → 근거 원문 위치 → 자료 공백·한계,6,3,기회와 위험 각각의 원문 근거가 충분한가요? / 각 수치가 어떤 원문·계산에서 나왔는지 근거 ID로 확인하세요.
6,L3-A,중급,적극형,comparative,사업 요약 → 경쟁 5요인 후보 발췌 → 기회 요인(성립 조건) → 위험 요인(하방 조건) → 근거 원문 위치 → 자료 공백·한계,0,3,이 기회가 성립하려면 어떤 조건이 계속 유지되어야 하나요? / 이 가설이 틀렸다고 판단할 근거(실패 조건)는 무엇인가요? / 반대 해석이 가능한 근거가 있는지 비교표로 확인하세요.
7,L3-C,중급,신중형,comparative,사업 요약 → 경쟁 5요인 후보 발췌 → 위험 요인(하방 조건) → 자료 공백·한계 → 기회 요인(성립 조건) → 근거 원문 위치,0,3,가장 큰 의존 위험(고객·원재료·규제)은 무엇이고 원문 근거가 있나요? / 위험을 완화하는 근거는 원문에서 확인되나요? / 반대 해석이 가능한 근거가 있는지 비교표로 확인하세요.
8,L3-U,중급,중립(미확정),comparative,사업 요약 → 경쟁 5요인 후보 발췌 → 기회 요인(성립 조건) → 위험 요인(하방 조건) → 근거 원문 위치 → 자료 공백·한계,0,3,기회와 위험 각각의 원문 근거가 충분한가요? / 반대 해석이 가능한 근거가 있는지 비교표로 확인하세요.
9,L4-A,고급,적극형,methodological,사업 요약 → 근거 원문 위치 → 자료 공백·한계 → 기회 요인(성립 조건) → 경쟁 5요인 후보 발췌 → 위험 요인(하방 조건),0,3,이 기회가 성립하려면 어떤 조건이 계속 유지되어야 하나요? / 이 가설이 틀렸다고 판단할 근거(실패 조건)는 무엇인가요? / 산식·기간·조정 기준을 바꾸면 결론이 달라지는지(민감도) 검토하세요.


## 2. 산업 Agent: 문단 분류 기준선
① Agent 근거 문단을 표로 내보냄 → ② 사람이 `human_label`과 `labeler` 입력 → ③ 읽어서 검증 → ④ 키워드 규칙 정확도 평가.
내보내는 표에는 규칙의 예측을 미리 넣지 않습니다(사람 판단이 끌려가지 않도록).

In [4]:
from agent.paragraph_labels import LABELS, export_sheet, paragraphs

_, industry = run(IndustryBusinessAgent, 'industry_business_model', AgentSettings(window=5))
display(pd.DataFrame(industry['paragraphClassification'])[['evidenceId', 'labelName', 'matched', 'reviewStatus']])
display(pd.DataFrame([{'코드': k, '분류 이름': v} for k, v in LABELS.items()]))

with tempfile.TemporaryDirectory() as folder:
    sheet = export_sheet(paragraphs(report), Path(folder) / 'labels.csv')
    display(pd.read_csv(sheet)[['evidenceId', 'subsection', 'text', 'human_label', 'labeler']])

,evidenceId,labelName,matched,reviewStatus
0,IND_E008,사업 개요,subsection:사업의 개요,unreviewed
1,IND_E010,경쟁·시장점유율,경쟁,unreviewed
2,IND_E012,원재료·공급·생산설비,원재료,unreviewed


,코드,분류 이름
0,business_overview,사업 개요
1,products_services,제품·서비스
2,revenue_orders,매출·수주
3,customers,고객·판매처
4,suppliers_materials,원재료·공급·생산설비
5,competition,경쟁·시장점유율
6,barriers_regulation,진입장벽·규제·특허
7,rnd_contracts,연구개발·주요계약
8,risk_management,위험관리·파생거래
9,other,기타·분류 불가


,evidenceId,subsection,text,human_label,labeler
0,IND_E008,1. 사업의 개요,모의 회사는 부품을 제조하여 완성품 업체에 공급합니다. 모의 문장입니다. 모의 문장입니다. 모의 문장입니다. 모의 문장입니다. 모의 문장입니다. 모의 문장입니다. 모의 문장입니다. 모의 문장입니다. 모의 문장...,NaN,NaN
1,IND_E010,2. 주요 제품 및 서비스,주요 고객은 완성품 업체이며 경쟁사와 가격 경쟁을 합니다.,NaN,NaN
2,IND_E012,3. 원재료 및 생산설비,주요 원재료는 외부 공급업체에서 매입합니다.,NaN,NaN


In [5]:
# 사람이 분류를 붙인 '예시' 정답셋(설명용으로 만든 가상 문단). 실제로는 load_labels('채운 CSV')를 씁니다.
from agent.paragraph_labels import evaluate, split_by_document

samples = [('경쟁사 대비 시장점유율이 높습니다', 'competition'), ('주요 원재료는 해외에서 매입합니다', 'suppliers_materials'),
           ('주요 고객은 완성차 업체입니다', 'customers'), ('당사는 1990년에 설립되었습니다', 'business_overview'),
           ('신규 수주가 늘었습니다', 'revenue_orders'), ('환율 변동 위험을 관리합니다', 'risk_management')]
gold = [{'evidenceId': f'E{d}_{i}', 'documentId': f'D{d}', 'documentVersion': 'v', 'publishedAt': f'2026-{d + 1:02d}-01',
         'text': text, 'subsection': '1. 사업의 개요' if label == 'business_overview' else None, 'human_label': label}
        for d in range(10) for i, (text, label) in enumerate(samples * 2)]
score = evaluate(gold)
print('문단 수', score['n'], '/ 정확도', round(score['accuracy'], 3), '/ macro F1', round(score['macroF1'], 3))
print('경고:', score['warnings'])
display(pd.DataFrame(score['perLabel']).T)
train, test = split_by_document(gold, 0.5)
print('문서 단위 분리 → 학습', len(train), '문단 / 평가', len(test), '문단 (평가 문서:', {r['documentId'] for r in test}, ')')

문단 수 120 / 정확도 1.0 / macro F1 1.0
경고: []


,precision,recall,f1,support
business_overview,1.0,1.0,1.0,20.0
revenue_orders,1.0,1.0,1.0,20.0
customers,1.0,1.0,1.0,20.0
suppliers_materials,1.0,1.0,1.0,20.0
competition,1.0,1.0,1.0,20.0
risk_management,1.0,1.0,1.0,20.0


문서 단위 분리 → 학습 60 문단 / 평가 60 문단 (평가 문서: {'D8', 'D6', 'D9', 'D7', 'D5'} )


## 3. 기술 Agent: 관측 상태 규칙
기존 계산값에 설명용 구간을 붙입니다. **예측·매매 신호가 아닙니다.** 임계값은 검증 전 초기 규칙입니다.

In [6]:
from agent.technical_states import THRESHOLDS

_, technical = run(TechnicalAnalysisAgent, 'technical_analysis', AgentSettings(window=5))
display(pd.DataFrame(technical['calculations'])[['calculationId', 'result', 'formula']])
display(pd.DataFrame([{'상태': c['claimId'], '판정': c['stateRule']['state'], '설명': c['text']}
                      for c in technical['claims'] if 'stateRule' in c]))
print('임계값:', THRESHOLDS)

,calculationId,result,formula
0,TECH_SMA,113.700000,mean(last window closes)
1,TECH_RETURN,0.013514,last close / previous close - 1
2,TECH_VOLATILITY,0.021966,population std of available simple returns
3,TECH_VOLUME_CHANGE,0.047619,last volume / mean(previous window volumes) - 1
4,TECH_VOLATILITY_RATIO,1.101753,pstdev(last window returns) / pstdev(all available returns)


,상태,판정,설명
0,TECH_STATE_TREND,이동평균 아래,"마지막 종가가 5개 관측 이동평균 대비 -1.06%로 ""이동평균 아래"" 상태입니다. (관측 상태 설명이며 예측·매매 신호가 아닙니다.)"
1,TECH_STATE_RETURN,작은 변동(상승),"직전 대비 수익률 +1.35%는 ""작은 변동(상승)"" 구간입니다. (관측 상태 설명이며 예측·매매 신호가 아닙니다.)"
2,TECH_STATE_VOLATILITY,전체 기간과 비슷한 변동성,"최근 5개 수익률 변동성이 전체 기간의 1.10배로 ""전체 기간과 비슷한 변동성""입니다. (관측 상태 설명이며 예측·매매 신호가 아닙니다.)"
3,TECH_STATE_VOLUME,평이한 거래량,"마지막 거래량이 직전 5개 평균 대비 +4.76%로 ""평이한 거래량""입니다. (관측 상태 설명이며 예측·매매 신호가 아닙니다.)"


임계값: {'nearSma': 0.01, 'returnBands': (0.03, 0.1), 'volatilityRatioBands': (0.67, 1.5), 'volumeBands': (-0.5, 0.5, 1.0)}


## 4. 평가: 페르소나 변경 전후 보존
12개 설정(L1~L4 × A/C/U)으로 다시 실행해 수치·근거·주장·품질 판정이 중립(L2-U)과 같은지 확인합니다.

In [7]:
from agent.persona_evaluation import evaluate_persona_invariance

for agent_cls, domain in [(IndustryBusinessAgent, 'industry_business_model'), (TechnicalAnalysisAgent, 'technical_analysis')]:
    gate, _ = run(agent_cls, domain, AgentSettings(window=5))
    table = evaluate_persona_invariance(report, gate, agent_cls, 5)
    print(domain, '→', table['status'])
    display(pd.DataFrame(table['rows']).drop(columns=['표현 차이']))

industry_business_model → passed


,페르소나,수치 보존,근거 보존,주장 보존,품질 판정 보존,보류·상태 보존,입력 보존,결과
0,L1-A,통과,통과,통과,통과,통과,통과,통과
1,L1-C,통과,통과,통과,통과,통과,통과,통과
2,L1-U,통과,통과,통과,통과,통과,통과,통과
3,L2-A,통과,통과,통과,통과,통과,통과,통과
4,L2-C,통과,통과,통과,통과,통과,통과,통과
5,L2-U,통과,통과,통과,통과,통과,통과,통과
6,L3-A,통과,통과,통과,통과,통과,통과,통과
7,L3-C,통과,통과,통과,통과,통과,통과,통과
8,L3-U,통과,통과,통과,통과,통과,통과,통과
9,L4-A,통과,통과,통과,통과,통과,통과,통과


technical_analysis → passed


,페르소나,수치 보존,근거 보존,주장 보존,품질 판정 보존,보류·상태 보존,입력 보존,결과
0,L1-A,통과,통과,통과,통과,통과,통과,통과
1,L1-C,통과,통과,통과,통과,통과,통과,통과
2,L1-U,통과,통과,통과,통과,통과,통과,통과
3,L2-A,통과,통과,통과,통과,통과,통과,통과
4,L2-C,통과,통과,통과,통과,통과,통과,통과
5,L2-U,통과,통과,통과,통과,통과,통과,통과
6,L3-A,통과,통과,통과,통과,통과,통과,통과
7,L3-C,통과,통과,통과,통과,통과,통과,통과
8,L3-U,통과,통과,통과,통과,통과,통과,통과
9,L4-A,통과,통과,통과,통과,통과,통과,통과


In [8]:
# 일부러 '적극형이면 수치를 10% 올리는' 잘못된 Agent를 만들면 평가표가 잡아냅니다.
class LeakyAgent(TechnicalAnalysisAgent):
    def run(self, report, **kwargs):
        result = super().run(report, **kwargs)
        if self.settings.style == 'A':
            result['calculations'][0]['result'] *= 1.1
        return result

gate, _ = run(TechnicalAnalysisAgent, 'technical_analysis', AgentSettings(window=5))
table = evaluate_persona_invariance(report, gate, LeakyAgent, 5)
print('잘못된 Agent 평가:', table['status'])
display(pd.DataFrame(table['rows'])[['페르소나', '수치 보존', '결과']])

잘못된 Agent 평가: failed


,페르소나,수치 보존,결과
0,L1-A,실패,실패
1,L1-C,통과,통과
2,L1-U,통과,통과
3,L2-A,실패,실패
4,L2-C,통과,통과
5,L2-U,통과,통과
6,L3-A,실패,실패
7,L3-C,통과,통과
8,L3-U,통과,통과
9,L4-A,실패,실패


## 5. ML 확장 판단
규칙 기준선에 약점이 있고, 같은 평가 정답셋에서 모델이 충분히 나을 때만 채택을 제안합니다. 여기서는 판단 규칙만 보여줍니다(모델 학습 없음).

In [9]:
from agent.model_comparison import compare, weaknesses

weak_rule = evaluate(test, predict=lambda text, sub: 'competition' if '경쟁' in text else 'other')
candidate = evaluate(test)  # '모델' 자리에 예시로 키워드 규칙 결과를 넣음
info = {'modelVersion': 'example-1', 'trainingCutoff': '2026-06-30', 'trainingGoldDigest': 'train-set-digest'}
print('기준선 약점:', weaknesses(weak_rule))
decision = compare(weak_rule, candidate, model_info=info)
print('채택 제안:', decision['adopt'], '/ macro F1 개선', round(decision['macroF1Gain'], 3))
for reason in decision['reasons']:
    print(' -', reason)

# 학습에 쓴 정답셋으로 평가하면(누수) 채택하지 않습니다.
leak = compare(weak_rule, candidate, model_info={**info, 'trainingGoldDigest': candidate['goldDigest']})
print('누수 사례 채택 제안:', leak['adopt'], leak['reasons'])

기준선 약점: [{'label': 'business_overview', 'f1': 0.0, 'support': 10}, {'label': 'revenue_orders', 'f1': 0.0, 'support': 10}, {'label': 'customers', 'f1': 0.0, 'support': 10}, {'label': 'suppliers_materials', 'f1': 0.0, 'support': 10}, {'label': 'risk_management', 'f1': 0.0, 'support': 10}]
채택 제안: True / macro F1 개선 0.833
 - 모든 채택 조건 충족(사람 최종 검토 필요)
누수 사례 채택 제안: False ['학습 정답셋과 평가 정답셋이 같음(누수)']
